In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
lever_bronze = spark.table(
    "job_market.bronze.lever_jobs"
)

print("Lever Bronze records:", lever_bronze.count())

lever_bronze.printSchema()

Lever Bronze records: 638
root
 |-- source: string (nullable = true)
 |-- company: string (nullable = true)
 |-- site: string (nullable = true)
 |-- ingestion_timestamp: string (nullable = true)
 |-- ingestion_date: string (nullable = true)
 |-- job_id: string (nullable = true)
 |-- job_title: string (nullable = true)
 |-- location: string (nullable = true)
 |-- all_locations: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- department: string (nullable = true)
 |-- team: string (nullable = true)
 |-- job_level: string (nullable = true)
 |-- employment_type: string (nullable = true)
 |-- description: string (nullable = true)
 |-- description_plain: string (nullable = true)
 |-- country: string (nullable = true)
 |-- workplace_type: string (nullable = true)
 |-- salary_range: struct (nullable = true)
 |    |-- currency: string (nullable = true)
 |    |-- interval: string (nullable = true)
 |    |-- max: long (nullable = true)
 |    |-- min: long (nullable = t

In [0]:
display(
    lever_bronze.select(
        "company",
        "job_id",
        "job_title",
        "location",
        "all_locations",
        "department",
        "team",
        "job_level",
        "employment_type",
        "workplace_type",
        "salary_range",
        "salary_description",
        "created_at"
    ).limit(20)
)

company,job_id,job_title,location,all_locations,department,team,job_level,employment_type,workplace_type,salary_range,salary_description,created_at
DEUNA,26b86395-02b2-4540-8d97-6c648f3fab79,AI Platform Tech Lead,San Francisco,List(San Francisco),null,Technology,null,Full time,hybrid,null,null,1779478818104
DEUNA,a1d9d480-6b02-448e-99a1-0e72e36b31d0,Backend Engineer,"Bogotá, Colombia","List(Bogotá, Colombia, Ciudad de México, Mexico, Argentina)",null,Technology,null,Full time,hybrid,"List(USD, per-year-salary, 0, 0)",null,1770830758239
DEUNA,b9d34afb-e1e8-4e87-8ec7-935838eb4f26,Engineering Manager,"Ciudad de México, Mexico","List(Ciudad de México, Mexico)",null,Technology,null,Full time,remote,null,null,1784836413644
DEUNA,194207ef-3ddf-4a7e-a9cc-3c11e2cf36a1,Engineering Manager AI,San Francisco,"List(San Francisco, Bogotá, Colombia, Ciudad de México, Mexico)",null,Technology,null,Full time,remote,null,null,1790629952691
DEUNA,03223f29-f268-4150-8cff-98e2609468fa,Front End - Product Manager,"Ciudad de México, Mexico","List(Ciudad de México, Mexico, Bogotá, Colombia)",null,Product,null,Full time,remote,null,null,1790103097503
DEUNA,e7fa7bdc-05c1-4ae7-a772-4187e171720a,Head of Finance,"Ciudad de México, Mexico","List(Ciudad de México, Mexico, Bogotá, Colombia)",null,Finance,null,Full time,remote,null,null,1763651255035
DEUNA,2bb6b2dc-5641-43ce-8065-21db8a82b8ac,Head of Implementation,San Francisco,List(San Francisco),null,Platforms Integrations,null,null,remote,null,null,1763650128497
DEUNA,bedf632f-77c7-4da8-a35b-1d850ad01cbe,Partnerships Manager US,San Francisco,"List(San Francisco, New York)",null,Partnerships,null,Full time,remote,null,null,1789767358698
DEUNA,ec4f69ac-f28e-496e-b925-45d21c53e467,Product Head of AI,United States,"List(United States, Bogotá, Colombia, Ciudad de México, Mexico)",null,Product,null,Full time,remote,null,null,1790266060094
DEUNA,6371fd3d-da9d-433e-ba9d-b9a72c6f6551,Product Head of POP,United States,"List(United States, Bogotá, Colombia, Ciudad de México, Mexico)",null,Product,null,Full time,remote,null,null,1790265363548


In [0]:
lever_silver = (
    lever_bronze
    .withColumn(
        "job_title",
        F.trim(
            F.regexp_replace(
                F.col("job_title"),
                r"\s+",
                " "
            )
        )
    )
)

In [0]:
lever_silver = (
    lever_silver
    .withColumn(
        "location",
        F.trim(
            F.regexp_replace(
                F.col("location"),
                r"\s+",
                " "
            )
        )
    )
)

In [0]:
lever_silver = (
    lever_silver
    .withColumn(
        "location_normalized",
        F.lower(
            F.trim(
                F.regexp_replace(
                    F.col("location"),
                    r"\s+",
                    " "
                )
            )
        )
    )
)

In [0]:
lever_silver = (
    lever_silver
    .withColumn(
        "department",
        F.when(
            F.col("department").isNotNull(),
            F.trim(
                F.regexp_replace(
                    F.col("department"),
                    r"\s+",
                    " "
                )
            )
        )
    )
)

In [0]:
lever_silver = (
    lever_silver
    .withColumn(
        "team",
        F.when(
            F.col("team").isNotNull(),
            F.trim(
                F.regexp_replace(
                    F.col("team"),
                    r"\s+",
                    " "
                )
            )
        )
    )
)

In [0]:
lever_silver = (
    lever_silver
    .withColumn(
        "job_level",
        F.when(
            F.col("job_level").isNotNull(),
            F.trim(
                F.regexp_replace(
                    F.col("job_level"),
                    r"\s+",
                    " "
                )
            )
        )
    )
)

In [0]:
lever_silver = (
    lever_silver
    .withColumn(
        "employment_type",
        F.when(
            F.col("employment_type").isNotNull(),
            F.trim(
                F.regexp_replace(
                    F.col("employment_type"),
                    r"\s+",
                    " "
                )
            )
        )
    )
)

In [0]:
lever_silver = (
    lever_silver
    .withColumn(
        "workplace_type",
        F.when(
            F.col("workplace_type").isNotNull(),
            F.lower(
                F.trim(
                    F.regexp_replace(
                        F.col("workplace_type"),
                        r"\s+",
                        " "
                    )
                )
            )
        )
    )
)

In [0]:
lever_silver = (
    lever_silver
    .withColumn(
        "is_remote",
        F.when(
            F.col("workplace_type") == "remote",
            F.lit(True)
        )
        .otherwise(F.lit(False))
    )
)

In [0]:
lever_silver = (
    lever_silver
    .withColumn(
        "description_plain",
        F.trim(F.col("description_plain"))
    )
)

In [0]:
lever_silver = (
    lever_silver
    .withColumn(
        "posted_at",
        F.to_timestamp(
            F.from_unixtime(
                F.col("created_at") / 1000
            )
        )
    )
)

In [0]:
display(
    lever_silver.select(
        "job_id",
        "job_title",
        "created_at",
        "posted_at"
    ).limit(20)
)

job_id,job_title,created_at,posted_at
26b86395-02b2-4540-8d97-6c648f3fab79,AI Platform Tech Lead,1779478818104,2026-05-22T19:40:18.000Z
a1d9d480-6b02-448e-99a1-0e72e36b31d0,Backend Engineer,1770830758239,2026-02-11T17:25:58.000Z
b9d34afb-e1e8-4e87-8ec7-935838eb4f26,Engineering Manager,1784836413644,2026-07-23T19:53:33.000Z
194207ef-3ddf-4a7e-a9cc-3c11e2cf36a1,Engineering Manager AI,1790629952691,2026-09-28T21:12:32.000Z
03223f29-f268-4150-8cff-98e2609468fa,Front End - Product Manager,1790103097503,2026-09-22T18:51:37.000Z
e7fa7bdc-05c1-4ae7-a772-4187e171720a,Head of Finance,1763651255035,2025-11-20T15:07:35.000Z
2bb6b2dc-5641-43ce-8065-21db8a82b8ac,Head of Implementation,1763650128497,2025-11-20T14:48:48.000Z
bedf632f-77c7-4da8-a35b-1d850ad01cbe,Partnerships Manager US,1789767358698,2026-09-18T21:35:58.000Z
ec4f69ac-f28e-496e-b925-45d21c53e467,Product Head of AI,1790266060094,2026-09-24T16:07:40.000Z
6371fd3d-da9d-433e-ba9d-b9a72c6f6551,Product Head of POP,1790265363548,2026-09-24T15:56:03.000Z


In [0]:
lever_silver = (
    lever_silver
    .withColumn(
        "salary_min",
        F.when(
            F.col("salary_range.min") > 0,
            F.col("salary_range.min")
        ).otherwise(F.lit(None))
    )
    .withColumn(
        "salary_max",
        F.when(
            F.col("salary_range.max") > 0,
            F.col("salary_range.max")
        ).otherwise(F.lit(None))
    )
    .withColumn(
        "salary_currency",
        F.col("salary_range.currency")
    )
    .withColumn(
        "salary_interval",
        F.col("salary_range.interval")
    )
)

In [0]:
display(
    lever_silver.select(
        "company",
        "job_title",
        "salary_min",
        "salary_max",
        "salary_currency",
        "salary_interval"
    )
    .filter(
        F.col("salary_min").isNotNull() |
        F.col("salary_max").isNotNull()
    )
    .limit(20)
)

company,job_title,salary_min,salary_max,salary_currency,salary_interval
Zoox,AV Safety Data Reporting Engineer,160000,221000,USD,per-year-salary
Zoox,Body Structures Engineer,170000,204000,USD,per-year-salary
Zoox,"Chassis Controls Engineer, Brakes & Steering",157000,245000,USD,per-year-salary
Zoox,Data Scientist - Perception Verification and Validation,167000,228000,USD,per-year-salary
Zoox,Diagnostics Serviceability Engineer,138000,192000,USD,per-year-salary
Zoox,"Director, Corporate Security",198000,282000,USD,per-year-salary
Zoox,"Director, Perception",410000,492000,USD,per-year-salary
Zoox,"Director, Product Management",335000,403000,USD,per-year-salary
Zoox,Embedded Software Engineer,164000,223000,USD,per-year-salary
Zoox,Embedded Software Engineer - Battery Management Systems,221000,265000,USD,per-year-salary


In [0]:
salary_errors = (
    lever_silver
    .filter(
        (F.col("salary_min").isNotNull()) &
        (F.col("salary_max").isNotNull()) &
        (F.col("salary_min") > F.col("salary_max"))
    )
)

print(
    "Salary records where min > max:",
    salary_errors.count()
)

display(
    salary_errors.select(
        "company",
        "job_title",
        "salary_min",
        "salary_max"
    )
)

Salary records where min > max: 0


company,job_title,salary_min,salary_max


In [0]:
salary_errors = (
    lever_silver
    .filter(
        (F.col("salary_min").isNotNull()) &
        (F.col("salary_max").isNotNull()) &
        (F.col("salary_min") > F.col("salary_max"))
    )
)

print(
    "Salary records where min > max:",
    salary_errors.count()
)

display(
    salary_errors.select(
        "company",
        "job_title",
        "salary_min",
        "salary_max"
    )
)

Salary records where min > max: 0


company,job_title,salary_min,salary_max


In [0]:
lever_silver.select(
    F.sum(
        F.col("job_id").isNull().cast("int")
    ).alias("null_job_id"),

    F.sum(
        F.col("job_title").isNull().cast("int")
    ).alias("null_job_title"),

    F.sum(
        F.col("company").isNull().cast("int")
    ).alias("null_company")
).show()

+-----------+--------------+------------+
|null_job_id|null_job_title|null_company|
+-----------+--------------+------------+
|          0|             0|           0|
+-----------+--------------+------------+



In [0]:
lever_silver = (
    lever_silver
    .filter(F.col("job_id").isNotNull())
    .filter(F.col("job_title").isNotNull())
    .filter(F.trim(F.col("job_title")) != "")
    .filter(F.col("company").isNotNull())
)

In [0]:
window_spec = (
    Window
    .partitionBy("job_id")
    .orderBy(
        F.col("posted_at").desc_nulls_last(),
        F.col("ingestion_timestamp").desc_nulls_last()
    )
)

lever_silver = (
    lever_silver
    .withColumn(
        "row_num",
        F.row_number().over(window_spec)
    )
    .filter(
        F.col("row_num") == 1
    )
    .drop("row_num")
)

In [0]:
lever_silver_final = lever_silver.select(

    # -------------------------
    # Lineage
    # -------------------------
    "source",
    "company",
    "ingestion_timestamp",
    "ingestion_date",

    # -------------------------
    # Job identity
    # -------------------------
    "job_id",
    "job_title",

    # -------------------------
    # Classification
    # -------------------------
    "department",
    "team",
    "job_level",
    "employment_type",

    # -------------------------
    # Location
    # -------------------------
    "location",
    "location_normalized",
    "all_locations",

    # -------------------------
    # Workplace
    # -------------------------
    "workplace_type",
    "is_remote",

    # -------------------------
    # Description
    # -------------------------
    "description_plain",

    # -------------------------
    # Salary
    # -------------------------
    "salary_min",
    "salary_max",
    "salary_currency",
    "salary_interval",

    # -------------------------
    # Date
    # -------------------------
    "posted_at",

    # -------------------------
    # URLs
    # -------------------------
    "job_url",
    "apply_url"
)

In [0]:
print(
    "Lever Silver records:",
    lever_silver_final.count()
)

Lever Silver records: 638


In [0]:
duplicate_jobs = (
    lever_silver_final
    .groupBy("job_id")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Duplicate job IDs:",
    duplicate_jobs.count()
)

display(duplicate_jobs)

Duplicate job IDs: 0


job_id,count


In [0]:
display(
    lever_silver_final
    .groupBy("company")
    .count()
    .orderBy(F.desc("count"))
)

company,count
Palantir,316
Zoox,229
Spotify,76
DEUNA,17


In [0]:
display(
    lever_silver_final
    .groupBy("workplace_type")
    .count()
    .orderBy(F.desc("count"))
)

workplace_type,count
hybrid,392
onsite,201
remote,45


In [0]:
display(
    lever_silver_final
    .select(
        F.count("*").alias("total_jobs"),
        F.sum(
            F.col("salary_min").isNotNull().cast("int")
        ).alias("jobs_with_salary")
    )
)

total_jobs,jobs_with_salary
638,215


In [0]:
display(
    lever_silver_final.select([
        F.sum(
            F.col(c).isNull().cast("int")
        ).alias(c)
        for c in lever_silver_final.columns
    ])
)

source,company,ingestion_timestamp,ingestion_date,job_id,job_title,department,team,job_level,employment_type,location,location_normalized,all_locations,workplace_type,is_remote,description_plain,salary_min,salary_max,salary_currency,salary_interval,posted_at,job_url,apply_url
0,0,0,0,0,0,333,0,612,14,0,0,0,0,0,0,423,423,422,422,0,0,0


In [0]:
(
    lever_silver_final
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "job_market.silver.lever_jobs"
    )
)

In [0]:
spark.sql("""
    SELECT *
    FROM job_market.silver.lever_jobs
    LIMIT 20
""").display()

source company ingestion_timestamp ingestion_date job_id job_title department team job_level employment_type location location_normalized all_locations workplace_type is_remote description_plain salary_min salary_max salary_currency salary_interval posted_at job_url apply_url lever Zoox 2026-10-06T12:52:06.606003+00:00 2026-10-06 000392a0-3844-47c4-b580-d56d9a97620c Engineering Manager, Onboard Execution Performance Software Software Systems null Full-time Foster City, CA foster city, ca List(Foster City, CA, Seattle, WA) hybrid false  
Zoox is building the world's most advanced self-driving hardware and software solution. The efficiency demands of such a system require an expert fine tuning of both the compute hardware architecture as well as the algorithms and middleware that runs on it to achieve maximum throughput at the most optimal power levels. 

The Software Performance team keeps the main onboard computer healthy and performant to ensure that our on-bot autonomy software meets its latency targets while also accommodating its safety goals. We are responsible for the efficient allocation of onboard compute resources (CPU/GPU/Memory/accelerators) ensuring contention is minimized and the autonomy stack executes harmoniously.

As the Onboard Execution team leader within the Performance team,  you will lead a highly specialized team of software performance engineers in aggressively optimizing on-bot software runtime. This is a high impact role that sits at the intersection of ML Platform, Core, and Autonomy Software - ensuring that efficient compute utilization is a central tenet emphasized across the organization. 
 
 
 237000 322000 USD per-year-salary 2026-04-17T15:58:31.000Z https://jobs.lever.co/zoox/000392a0-3844-47c4-b580-d56d9a97620c https://jobs.lever.co/zoox/000392a0-3844-47c4-b580-d56d9a97620c/apply lever Zoox 2026-10-06T12:52:06.606003+00:00 2026-10-06 00bede24-f74d-4447-ace1-48638965382e Senior / Staff Software Engineer - Motion Planning Software Autonomy Software null Full-time Foster City, CA foster city, ca List(Foster City, CA, Boston, MA) hybrid false On the Motion Planning team at Zoox, you’ll be dedicated to improving the driving behaviors of the robot on public roads. The Planner's job is to figure out where the robot is going and how to get there safely in situations as complex as those found in Downtown San Francisco or on the Las Vegas Strip.
 
This is a cutting-edge problem, which requires teamwork and collaboration across a broad set of experts spanning Planner, Perception, Prediction, Operations, and more. Together, you will create the next generation of algorithmic improvements to expand our robot’s ability to drive in some of the most complex situations found on public roads.
 
Safety is foundational at Zoox, so you will take a measured approach – develop a principled modification to the Planner, test in simulation and on closed courses, and team up with our safety drivers for on-road testing. 
 254000 350000 USD per-year-salary 2025-11-05T20:41:44.000Z https://jobs.lever.co/zoox/00bede24-f74d-4447-ace1-48638965382e https://jobs.lever.co/zoox/00bede24-f74d-4447-ace1-48638965382e/apply lever Palantir 2026-10-06T12:52:06.606003+00:00 2026-10-06 00c2c97b-8514-4617-9883-e53e486b6dcd Forward Deployed Enablement Engineer - Customer Success null Engineering null Full-time London, United Kingdom london, united kingdom List(London, United Kingdom) hybrid false A World-Changing Company
 
Palantir builds the world’s leading software for data-driven decisions and operations. By bringing the right data to the people who need it, our platforms empower our partners to develop lifesaving drugs, forecast supply chain disruptions, locate missing children, and more.


The Role
 
Forward Deployed Enablement Engineers are embedded with centralised customer success teams to maximise the outcomes of our deployed products and workflows across all commercial customers from small-scale start-ups to large enterprises. Thi

In [0]:
spark.sql("""
    SELECT
        company,
        COUNT(*) AS job_count
    FROM job_market.silver.lever_jobs
    GROUP BY company
    ORDER BY job_count DESC
""").display()

company,job_count
Palantir,316
Zoox,229
Spotify,76
DEUNA,17
